In [45]:
import numpy as np 
import matplotlib.pyplot as plt
import OrcFxAPI
import h5py
import pandas as pd
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.optimize import minimize_scalar



In [46]:
model_path = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Decay_opzet_harlequin.dat"
exp_path = r"C:\Users\verav\Desktop\Studie\Afstuderen\Decay_Data\01_Rawdata\02\002\34224_03CB_02_002_008_01_Decay1.h5m"

lookback_window = 5.0      # seconden vóór decay-start waarin we de amplitude zoeken
quiet_window_end = 12.0     # rustige periode eindigt 12 s vóór decay-start
quiet_window_length = 50.0


In [47]:
def run_decay_case(model_path, A0, exp_time, exp_signal, delta_t_exp,
                   linear_damping, quadratic_damping):
    """
    Draait 1 OrcaFlex decay case en vergelijkt met experiment.
    """

    # -------------------------
    # model opnieuw openen
    # -------------------------
    model = OrcFxAPI.Model(model_path)
    constraint = model["decay_constraint"]
    floaters = model["floaters"]
    floatertype = model["Floatertype"]

    # -------------------------
    # beginverplaatsing zetten
    # -------------------------


    # -------------------------
    constraint.InFrameInitialZ = A0
    constraint.InFrameInitialX = 0
    constraint.InFrameInitialY = 0
    constraint.InFrameInitialAzimuth = 0
    constraint.InFrameInitialDeclination = 0 
    constraint.InFrameInitialGamma = 0

    floatertype.OtherDampingLinearCoeffx = 0 #surge
    floatertype.OtherDampingLinearCoeffy = 0 #sway
    floatertype.OtherDampingLinearCoeffz  = linear_damping
    floatertype.OtherDampingLinearCoeffRx = 0 #roll
    floatertype.OtherDampingLinearCoeffRy = 0 #pitch
    floatertype.OtherDampingLinearCoeffRz = 0 #yaw

    floatertype.OtherDampingQuadraticCoeffx = 0 #surge
    floatertype.OtherDampingQuadraticCoeffy = 0 #sway 
    floatertype.OtherDampingQuadraticCoeffz = quadratic_damping #heav
    floatertype.OtherDampingQuadraticCoeffRx = 0#roll
    floatertype.OtherDampingQuadraticCoeffRy = 0 #pitch
    floatertype.OtherDampingQuadraticCoeffRz = 0 #yaw
        # run
    # -------------------------
    model.RunSimulation()

    
    t_sim = model.general.TimeHistory("Time")
    z_sim = floaters.TimeHistory("Z")

    # -------------------------
    # tijd verschuiven met experimenteel verschil
    # -------------------------
    t_sim_decay = t_sim - t_sim[0]
    t_sim_shifted = t_sim_decay - delta_t_exp

    # -------------------------
    # interpolatiefunctie
    # -------------------------
    f_sim = interp1d(
        t_sim_shifted,
        z_sim,
        bounds_error=False,
        fill_value=np.nan
    )

    # -------------------------
    # kleine extra shift fitten
    # -------------------------
    def compute_rmse(time_shift):
        z_sim_trial = f_sim(exp_time + time_shift)
        mask = ~np.isnan(z_sim_trial)

        if np.sum(mask) < 10:
            return 1e9

        error = z_sim_trial[mask] - exp_signal[mask]
        return np.sqrt(np.mean(error**2))

    result = minimize_scalar(
        compute_rmse,
        bounds=(-2.0, 2.0),
        method="bounded"
    )

    best_shift = result.x
    z_sim_fit = f_sim(exp_time + best_shift)

    mask = ~np.isnan(z_sim_fit)

    error = z_sim_fit[mask] - exp_signal[mask]
    rmse = np.sqrt(np.mean(error**2))
    mae = np.mean(np.abs(error))

    return {
        "linear_damping": linear_damping,
        "quadratic_damping": quadratic_damping,
        "best_time_shift": best_shift,
        "rmse": rmse,
        "mae": mae,
        "t_sim_shifted": t_sim_shifted,
        "z_sim": z_sim,
        "z_sim_fit": z_sim_fit,
        "mask": mask,
    }

In [48]:
with h5py.File(exp_path, "r") as f:
    t_exp = f["CroppedSignals/time"][:]
    z_exp = f["CroppedSignals/Z_COG (LPF: 5.0 rad*s^-1)"][:]
    t_unfiltered = f["UnfilteredSignals/time"][:]
    z_unfiltered = f["UnfilteredSignals/Z_COG (unfiltered)"][:]

# DataFrames in één keer maken
df_exp = pd.DataFrame({
    "time [s]": t_exp,
    "time_norm [s]": t_exp - t_exp[0],
    "z_exp [m]": z_exp
})

df_unfiltered = pd.DataFrame({
    "time [s]": t_unfiltered,
    "z_unfiltered [m]": z_unfiltered
})


In [49]:
t_decay_start = t_exp[0]

# -----------------------------
# 2. rustige periode in ongefilterd signaal kiezen
#    van (start - quiet_window_end - quiet_window_length) tot (start - quiet_window_end)
# -----------------------------
quiet_start = t_decay_start - quiet_window_end - quiet_window_length
quiet_end = t_decay_start - quiet_window_end

mask_quiet = (t_unfiltered >= quiet_start) & (t_unfiltered <= quiet_end)

if not np.any(mask_quiet):
    raise ValueError("Geen data gevonden in de gekozen rustige periode. Kies andere vensters.")

z_eq = np.mean(z_unfiltered[mask_quiet])

# -----------------------------
# 3. zoekvenster voor initiële amplitude
#    10 s vóór decay-start
# -----------------------------
search_start = t_decay_start - lookback_window
search_end = t_decay_start

mask_search = (t_unfiltered >= search_start) & (t_unfiltered <= search_end)

if not np.any(mask_search):
    raise ValueError("Geen data gevonden in het zoekvenster vóór de decay-start.")

t_search = t_unfiltered[mask_search]
z_search = z_unfiltered[mask_search]

# afwijking t.o.v. evenwicht
deviation = z_search - z_eq

# index van grootste absolute uitslag
idx_local = np.argmax(np.abs(deviation))

t_init = t_search[idx_local]
z_init = z_search[idx_local]
A0 = np.abs(z_init - z_eq)

delta_t_exp = t_decay_start - t_init
t_exp_decay = t_exp - t_decay_start

print(f"Decay start time           = {t_decay_start:.3f} s")
print(f"Quiet window               = [{quiet_start:.3f}, {quiet_end:.3f}] s")
print(f"Equilibrium level z_eq     = {z_eq:.6f} m")
print(f"Search window              = [{search_start:.3f}, {search_end:.3f}] s")
print(f"Initial extreme at t       = {t_init:.3f} s")
print(f"Initial extreme z          = {z_init:.6f} m")
print(f"Initial amplitude A0       = {A0:.6f} m")
print(f"Signed amplitude           = {z_init - z_eq:.6f} m")

Decay start time           = 324.059 s
Quiet window               = [262.059, 312.059] s
Equilibrium level z_eq     = 0.018519 m
Search window              = [319.059, 324.059] s
Initial extreme at t       = 320.922 s
Initial extreme z          = -5.307183 m
Initial amplitude A0       = 5.325702 m
Signed amplitude           = -5.325702 m


In [50]:
target_rmse = 0.20
target_mae = 0.15

c_lin = 1
c_quad =1

step_lin = 0.2
step_quad = 0.2

min_step_lin = 0.05
min_step_quad = 0.05

max_iter = 40

result = run_decay_case(
    model_path=model_path,
    A0=A0,
    exp_time=t_exp_decay,
    exp_signal=z_exp,
    delta_t_exp=delta_t_exp,
    linear_damping=c_lin,
    quadratic_damping=c_quad
)

print("RMSE =", result["rmse"])
print("MAE  =", result["mae"])



RMSE = 1.9448235069366935
MAE  = 1.7032587116386695


In [51]:
def evaluate_case(c_lin, c_quad):
    result = run_decay_case(
        model_path=model_path,
        A0=A0,
        exp_time=t_exp_decay,
        exp_signal=z_exp,
        delta_t_exp=delta_t_exp,
        linear_damping=c_lin,
        quadratic_damping=c_quad
    )
    return result

In [52]:
def random_candidates(n_samples, lin_range, quad_range):
    candidates = []

    for _ in range(n_samples):
        lin = np.random.uniform(lin_range[0], lin_range[1])
        quad = np.random.uniform(quad_range[0], quad_range[1])

        candidates.append((lin, quad))

    return candidates

In [53]:
def evaluate_random_samples(n_samples, lin_range, quad_range):
    candidates = random_candidates(n_samples, lin_range, quad_range)

    results = []

    for lin, quad in candidates:
        print(f"Testing lin={lin:.2f}, quad={quad:.2f}")

        res = evaluate_case(lin, quad)

        results.append({
            "linear_damping": lin,
            "quadratic_damping": quad,
            "rmse": res["rmse"],
            "mae": res["mae"],
            "best_time_shift": res["best_time_shift"]
        })

    return pd.DataFrame(results)

In [54]:
df_random = evaluate_random_samples(
    n_samples=30,
    lin_range=(0, 5),
    quad_range=(2, 4)
)

Testing lin=3.57, quad=3.50
Testing lin=1.57, quad=2.75
Testing lin=0.07, quad=2.84
Testing lin=3.23, quad=3.84
Testing lin=2.92, quad=3.93
Testing lin=3.18, quad=2.04
Testing lin=1.69, quad=3.85
Testing lin=2.32, quad=2.98
Testing lin=0.77, quad=3.01
Testing lin=0.78, quad=2.52
Testing lin=3.56, quad=2.72
Testing lin=4.17, quad=2.13
Testing lin=2.36, quad=3.00
Testing lin=3.56, quad=3.49
Testing lin=3.64, quad=3.39
Testing lin=1.29, quad=2.51
Testing lin=0.18, quad=2.23
Testing lin=1.88, quad=3.61
Testing lin=3.09, quad=3.76
Testing lin=4.68, quad=2.67
Testing lin=1.90, quad=3.02
Testing lin=2.39, quad=3.34
Testing lin=0.25, quad=3.99
Testing lin=1.88, quad=3.16
Testing lin=3.01, quad=3.75
Testing lin=2.27, quad=2.43
Testing lin=2.80, quad=3.52
Testing lin=1.15, quad=2.36
Testing lin=3.99, quad=2.33
Testing lin=4.33, quad=2.84


In [55]:
df_random = df_random.sort_values("rmse").reset_index(drop=True)
df_random.head(10)

,linear_damping,quadratic_damping,rmse,mae,best_time_shift
0,3.232876,3.840678,1.083946,0.940847,0.781564
1,2.916998,3.926686,1.086796,0.943355,0.781564
2,3.086638,3.755938,1.094261,0.949376,0.781560
3,3.010869,3.752214,1.096776,0.951480,0.781563
4,3.566445,3.497958,1.100041,0.953819,0.781561
5,3.561244,3.492817,1.100624,0.954290,0.781562
6,3.642969,3.391983,1.106557,0.958999,0.781562
7,2.800821,3.521644,1.122513,0.972506,0.781564
8,1.692807,3.849996,1.129882,0.979053,0.779649
9,4.325980,2.841490,1.137143,0.983294,0.780204
